# OCR Kartu Identitas

## Persiapan

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import csv
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from PIL.ExifTags import TAGS
from scipy.ndimage import laplace
from scipy.ndimage import median_filter
import re
import hashlib



df = pd.read_csv("ground_truth.csv")

In [ ]:
df.head()

e.. kayanya ada formatting yang aneh dari csvnya.

In [ ]:
with open("ground_truth.csv", "r", encoding="utf-8-sig") as f:
    for i, line in enumerate(f):
        print(f"{i+1}: {line.rstrip()}")
        if i >= 20:
            break

oh, kok ada petiknya. pantes. semuanya kah?

In [ ]:
with open("ground_truth.csv", "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

comma_counts = Counter()
outer_quoted = 0
normal = 0

for line in lines[1:]: 
    line = line.rstrip("\n\r")

    comma_count = line.count(",")
    comma_counts[comma_count] += 1

    if line.startswith('"'):
        outer_quoted += 1
    else:
        normal += 1

print(f"Total baris      : {len(lines) - 1}")
print(f"Dibungkus petik  : {outer_quoted}")
print(f"Format biasa     : {normal}")

print("\nJumlah koma:")
for count, total in sorted(comma_counts.items()):
    print(f"{count:>3} commas : {total} rows")

ngga semua, harus dihandle kedua kondisinya sih. kalo yang komanya banyak, bisa jadi dari alamatnya. nanti harus tetep dicek sih.

In [ ]:
def parse_ground_truth_row(line):
    line = line.rstrip("\r\n")

    # Beberapa baris dibungkus petik tambahan.
    if line.startswith("\"") and line.endswith("\""):
        line = line[1:-1].replace("\"\"", "\"")

    return next(csv.reader([line]))


with open("ground_truth.csv", "r", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)

with open("ground_truth.csv", "r", encoding="utf-8-sig") as f:
    next(f)
    rows = [parse_ground_truth_row(line) for line in f if line.strip()]

invalid_rows = [(i + 2, row) for i, row in enumerate(rows) if len(row) != len(header)]
if invalid_rows:
    raise ValueError(f"Ada baris dengan jumlah field yang aneh: {invalid_rows[:5]}")

df = pd.DataFrame(rows, columns=header)
print(f"Berhasil memuat {len(df)} baris dengan kolom: {list(df.columns)}")
df.head()

In [ ]:
df = df.replace("", pd.NA)

In [ ]:
field_counts = Counter()

for _, row in df.iterrows():
    field_counts[row.notna().sum()] += 1

print(f"Total baris      : {len(df)}")
print(f"Total columns    : {len(df.columns)}")

print("\nFields per row:")
for fields, total in sorted(field_counts.items()):
    print(f"{fields:>3} fields : {total} rows")

nice, udah 4 kolom semua. untuk sementara, ini dulu buat ground truth. next ke gambar bentar.

In [ ]:

images_dir = Path("images")
file_count = sum(path.is_file() for path in images_dir.iterdir())
print(f"Jumlah file di folder {images_dir}/: {file_count}")

e.. ground truth ada 632 tapi gambarnya ada 732?

## EDA Ground Truth

In [ ]:
df.isna().sum()

### Cek nilai unik setiap field

In [ ]:
print("Baris yang duplikat berdasarkan filename:")
display(df[df.duplicated("filename", keep=False)].sort_values("filename"))
print("\nJumlah nilai unik di setiap field:")
display(df.nunique())

### Cek keseragaman format nama

In [ ]:
name_check = df["name"].fillna("").astype(str)
def cek_nama(value):
    if not value.strip():
        return "kosong"
    if any(char.isalpha() and not char.isascii() for char in value):
        return "huruf beraksen/non-ASCII"
    if any(not (char.isascii() and char.isupper()) and char != " " for char in value):
        return "simbol atau angka"
    if value != value.upper():
        return "ada huruf kecil"
    if value != value.strip() or "  " in value:
        return "whitespace"
    return "format oke"

name_reasons = name_check.map(cek_nama)
print("Alasan format nama:")
display(name_reasons.value_counts().rename_axis("alasan").to_frame("jumlah"))
display(df.assign(alasan=name_reasons).query("alasan != 'format oke'")[['filename', 'name', 'alasan']])

name_reasons.value_counts().plot(kind="bar", title="Ringkasan format nama", ylabel="Jumlah", xlabel="Alasan", figsize=(8, 4))
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

### Cek format tanggal lahir

In [ ]:
birth_check = df["birth_date"].fillna("").astype(str)
birth_issues = pd.to_datetime(birth_check, format="%Y-%m-%d", errors="coerce").isna()
print(f"Jumlah tanggal lahir dengan format yang aneh: {birth_issues.sum()}")
display(df.loc[birth_issues, ["filename", "birth_date"]])

In [ ]:
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
matches_pattern = df["birth_date"].apply(lambda x: bool(date_pattern.match(str(x))))

print(f"Match pola YYYY-MM-DD : {matches_pattern.sum()} / {len(df)}")

date_parts = df.loc[matches_pattern, "birth_date"].str.split("-", expand=True)
date_parts.columns = ["Y", "M", "D"]
date_parts = date_parts.astype(int)

print(f"\nRange bagian ke-2 (posisi bulan): {date_parts['M'].min()}-{date_parts['M'].max()}")
print(f"Range bagian ke-3 (posisi hari)  : {date_parts['D'].min()}-{date_parts['D'].max()}")

swapped_suspect = date_parts[date_parts["M"] > 12]
print(f"\nBaris dengan bagian ke-2 >12 (indikasi tertukar) : {len(swapped_suspect)}")

### Cek formatting alamat

In [ ]:
address_check = df["address"].fillna("").astype(str)
def cek_alamat(value):
    if not value.strip():
        return "kosong"
    if value != value.strip():
        return "spasi di awal/akhir"
    if '"' in value:
        return "ada petik"
    return "format oke"

address_reasons = df["address"].fillna("").astype(str).map(cek_alamat)
print("Alasan format alamat:")
display(address_reasons.value_counts().rename_axis("alasan").to_frame("jumlah"))
display(df.assign(alasan=address_reasons).query("alasan != 'format oke'")[['filename', 'address', 'alasan']])

address_reasons.value_counts().plot(kind="bar", title="Ringkasan format alamat", ylabel="Jumlah", xlabel="Alasan", figsize=(8, 4))
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
vocab_summary = []
for field in ["name", "birth_date", "address"]:
    vocabulary = Counter("".join(df[field].fillna("").astype(str)))
    vocab_summary.append({"field": field, "jumlah karakter unik": len(vocabulary), "karakter paling sering": vocabulary.most_common(5)})
    print(f"\n{field}: ada {len(vocabulary)} karakter unik")
    print(sorted(vocabulary.items(), key=lambda item: (-item[1], item[0])))

vocab_summary = pd.DataFrame(vocab_summary)
print("Summary vocabulary:")
display(vocab_summary)
vocab_summary.plot.bar(x="field", y="jumlah karakter unik", legend=False, title="Jumlah karakter unik per field", ylabel="Jumlah karakter", figsize=(8, 4))
plt.tight_layout(); plt.show()

In [ ]:
# cek karakter yang di luar ASCII
special_chars = []

for field in ["name", "birth_date", "address"]:
    vocabulary = Counter(
        "".join(df[field].fillna("").astype(str))
    )

    for char, count in vocabulary.items():
        if not char.isascii():
            special_chars.append({
                "field": field,
                "character": char,
                "count": count
            })

special_chars_df = pd.DataFrame(special_chars)

print("Karakter non-ASCII:")
display(
    special_chars_df.sort_values(
        ["field", "count"],
        ascending=[True, False]
    )
)

In [ ]:
length_summary = pd.DataFrame({field: df[field].fillna("").astype(str).str.len() for field in ["name", "birth_date", "address"]}).describe().T
display(length_summary)

### Cek Duplikat Identitas

belum diselesein sebelumnya, name/birth_date/address ada yang duplikat (132 nama unik dari 632 baris). mau cek beneran itu 1 orang yang difoto berkali-kali (kartu sama atau kartu beda), bukan salah assign ground truth. cek dulu jumlah baris per identitas.

In [ ]:
identity_key = list(zip(df["name"], df["birth_date"], df["address"]))
df = df.assign(identity_key=identity_key)

identity_counts = df["identity_key"].value_counts()

print(f"Jumlah identitas unik : {len(identity_counts)}")
print(f"Total baris           : {len(df)}")
print("\nDistribusi jumlah gambar per identitas:")
display(identity_counts.value_counts().sort_index().rename_axis("jumlah_gambar").to_frame("jumlah_identitas"))

identity_counts.value_counts().sort_index().plot(
    kind="bar", title="Jumlah gambar per identitas", xlabel="Jumlah gambar", ylabel="Jumlah identitas", figsize=(8, 4)
)
plt.tight_layout(); plt.show()

sekarang liat langsung gambarnya buat beberapa identitas yang punya >1 baris. tujuannya bukan buat cek kualitas gambar (itu nanti pas EDA image), tapi buat konfirmasi: apakah emang orang yang sama, kartu yang sama difoto ulang, atau ada yang aneh (misal ground truth ke-assign salah).

In [ ]:
images_dir = Path("images")


def show_identity_group(filenames, label="", max_size=800):
    n = len(filenames)

    fig, axes = plt.subplots(
        1, n,
        figsize=(4 * n, 4)
    )

    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for ax, fname in zip(axes, filenames):
        img_path = images_dir / fname

        if img_path.exists():
            # Buka gambar lalu resize sebelum diberikan ke matplotlib
            with Image.open(img_path) as img:
                img.thumbnail((max_size, max_size))
                img = img.convert("RGB").copy()

            ax.imshow(img)
        else:
            ax.text(
                0.5,
                0.5,
                "gambar tidak ditemukan",
                ha="center",
                va="center"
            )

        ax.set_title(fname, fontsize=9)
        ax.axis("off")

    fig.suptitle(label, fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# ambil beberapa contoh identitas yang punya >1 gambar, dari yang jumlahnya paling banyak
duplicate_identities = identity_counts[identity_counts > 1].sort_values(ascending=False)
print(f"Jumlah identitas dengan >1 gambar: {len(duplicate_identities)}")
display(duplicate_identities.head(10))

sample_keys = duplicate_identities.head(3).index.tolist()

for key in sample_keys:
    name, birth_date, address = key
    filenames = df.loc[df["identity_key"] == key, "filename"].tolist()
    show_identity_group(filenames, label=f"{name} | {birth_date}")

duplikat identitas memang adalah hasil dari data gambar dari id card yang sama, namun dengan kondisi yang berbeda beda.

## 1. EDA Image Dataset

### a. Load & Inventarisasi File

In [ ]:
images_dir = Path("images")
image_files = sorted(path for path in images_dir.iterdir() if path.is_file())

print(f"Total file di {images_dir}/: {len(image_files)}")
print("\nDaftar file:")
for path in image_files:
    print(path.name)

### b. Cek File Corrupt / Gagal Dibuka

In [ ]:
corrupt_files = []
image_records = []

for path in image_files:
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            image_records.append({
                "filename": path.name,
                "extension": path.suffix.lower(),
                "actual_format": image.format,
                "width": image.width,
                "height": image.height,
                "mode": image.mode,
            })
    except Exception as error:
        corrupt_files.append({"filename": path.name, "error": str(error)})

print(f"File berhasil dibuka: {len(image_records)}")
print(f"File corrupt/gagal dibuka: {len(corrupt_files)}")
if corrupt_files:
    display(pd.DataFrame(corrupt_files))
else:
    print("Tidak ada file yang corrupt atau gagal dibuka.")

image_df = pd.DataFrame(image_records)

**Catatan penting** : image_df disini menggunakan keseluruhan gambar, bukan hanya gambar yang sesuai dengan ground truth.

### c. Cek Ekstensi vs Format Asli

In [ ]:
format_to_extensions = {
    "JPEG": {".jpg", ".jpeg"},
    "PNG": {".png"},
    "WEBP": {".webp"},
    "GIF": {".gif"},
    "BMP": {".bmp"},
    "TIFF": {".tif", ".tiff"},
}

image_df["format_mismatch"] = image_df.apply(
    lambda row: row["extension"] not in format_to_extensions.get(row["actual_format"], set()),
    axis=1,
)

mismatches = image_df[image_df["format_mismatch"]]
print(f"Format ekstensi yang mismatch: {len(mismatches)}")
if len(mismatches):
    display(mismatches[["filename", "extension", "actual_format"]])
else:
    print("Tidak ada mismatch antara ekstensi dan format asli.")

semua file aman, gada yang corrupt atau apa

## 2. Cross-check Filename vs Ground Truth

Bandingkan filename yang ada di folder `images/` dengan filename di `ground_truth.csv`.

### a. Gambar yang Tidak Ada di Ground Truth

In [ ]:
image_filenames = {path.name for path in image_files}
ground_truth_filenames = set(df["filename"].dropna().astype(str).str.strip())

images_not_in_ground_truth = sorted(image_filenames - ground_truth_filenames)
print(f"Gambar di folder tapi tidak ada di ground truth: {len(images_not_in_ground_truth)}")
print(images_not_in_ground_truth)

yep, sesuai eda awal tadi, ada 100 gambar "lebih" dari ground truth.

### b. Gambar Ground Truth yang Tidak Ditemukan

In [ ]:
ground_truth_without_image = sorted(ground_truth_filenames - image_filenames)
print(f"Filename di ground truth tapi gambarnya tidak ditemukan: {len(ground_truth_without_image)}")
print(ground_truth_without_image)

if len(ground_truth_without_image) == 0:
    print("Ideal: semua filename di ground truth punya gambar yang sesuai.")

tapi dari semua yang ada di ground truth ada gambar aslinya

### c. Cek Pola Nama File

In [ ]:
filename_pattern = re.compile(r"^image_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)
filename_pattern_issues = []

for filename in sorted(image_filenames | ground_truth_filenames):
    match = filename_pattern.fullmatch(filename)
    if match is None:
        filename_pattern_issues.append({"filename": filename, "alasan": "tidak mengikuti pola image_NNN.jpg/png"})

print(f"Filename dengan pola yang menyimpang: {len(filename_pattern_issues)}")
if filename_pattern_issues:
    display(pd.DataFrame(filename_pattern_issues))
else:
    print("Semua filename mengikuti pola image_NNN.jpg/jpeg/png.")

numbers = [int(filename_pattern.fullmatch(filename).group(1)) for filename in image_filenames if filename_pattern.fullmatch(filename)]
print(f"Rentang nomor gambar: {min(numbers)} sampai {max(numbers)}")
missing_numbers = sorted(set(range(min(numbers), max(numbers) + 1)) - set(numbers))
print(f"Nomor yang hilang di tengah urutan: {missing_numbers}")

format penamaan file aman semua

## 3. Duplikat File (Byte-level)

### a. Hash MD5 Tiap File

In [ ]:
def md5_file(path, chunk_size=1024 * 1024):
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

file_hashes = pd.DataFrame({
    "filename": [path.name for path in image_files],
    "md5": [md5_file(path) for path in image_files],
})
file_hashes.head()

### b. Kelompok File dengan Hash Sama

In [ ]:
duplicate_hash_groups = (
    file_hashes.groupby("md5")["filename"]
    .agg(list)
    .loc[lambda series: series.str.len() > 1]
)

print(f"Jumlah grup file byte-identik: {len(duplicate_hash_groups)}")
print(f"Jumlah file yang termasuk grup duplikat: {duplicate_hash_groups.map(len).sum() if len(duplicate_hash_groups) else 0}")
display(duplicate_hash_groups.rename("files").to_frame())

### c. Bandingkan dengan Duplikat Identitas

In [ ]:
identity_cols = ["name", "birth_date", "address"]
identity_df = df[["filename"] + identity_cols].copy()
identity_df["identity_key"] = identity_df[identity_cols].fillna("").astype(str).agg(" | ".join, axis=1)
identity_duplicate_groups = (
    identity_df.groupby("identity_key")["filename"]
    .agg(list)
    .loc[lambda series: series.str.len() > 1]
)

hash_with_identity = file_hashes.merge(identity_df[["filename", "identity_key"]], on="filename", how="left")
hash_groups_with_identity = (
    hash_with_identity.groupby("md5").agg({"filename": list, "identity_key": lambda values: set(values)})
    .loc[lambda frame: frame["filename"].str.len() > 1]
)
hash_groups_with_identity["kategori"] = hash_groups_with_identity["identity_key"].map(
    lambda identities: "overlap dengan duplikat identitas" if len(identities) == 1 else "kasus baru: byte sama, identitas berbeda"
)

print(f"Grup duplikat identitas: {len(identity_duplicate_groups)}")
print("Perbandingan grup byte-identik:")
display(hash_groups_with_identity.reset_index()[["md5", "filename", "identity_key", "kategori"]])

founding: tidak ada gambar duplikat

## 4. Properti Format, Resolusi & Metadata

Metadata hanya untuk audit/EDA, bukan input tambahan ke pipeline OCR.

### a. Ekstensi & Format Gambar

In [ ]:
display(image_df['extension'].value_counts().rename('jumlah').to_frame())
display(image_df['actual_format'].value_counts().rename('jumlah').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
image_df['extension'].value_counts().plot.bar(ax=axes[0], title='Ekstensi')
image_df['actual_format'].value_counts().plot.bar(ax=axes[1], title='Format asli')
plt.tight_layout(); plt.show()

### b. Color Mode

In [ ]:
display(image_df['mode'].value_counts().rename('jumlah').to_frame())
image_df['mode'].value_counts().plot.bar(title='Distribusi color mode', ylabel='Jumlah', figsize=(7, 4))
plt.tight_layout(); plt.show()

semua file image berupa jpg dengan tipe format asli berupa jpeg semua. dan semua gambar berada menggunakan color mode RGB.

### c. Width × Height

In [ ]:
image_df['pixels'] = image_df['width'] * image_df['height']
display(image_df[['width', 'height', 'pixels']].describe())
plt.figure(figsize=(7, 5)); plt.scatter(image_df['width'], image_df['height'], alpha=0.6); plt.title('Scatter plot resolusi'); plt.xlabel('Width'); plt.ylabel('Height'); plt.tight_layout(); plt.show()

### d. Aspect Ratio

In [ ]:
image_df['aspect_ratio'] = image_df['width'] / image_df['height']
display(image_df['aspect_ratio'].describe().to_frame().T)

rasio_umum = {'1:1': 1, '4:3': 4 / 3, '3:4': 3 / 4, '3:2': 3 / 2, '2:3': 2 / 3, '16:9': 16 / 9, '9:16': 9 / 16}

def kelompokkan_rasio(value, tolerance=0.03):
    nama_rasio, nilai_rasio = min(rasio_umum.items(), key=lambda item: abs(value - item[1]))
    return nama_rasio if abs(value - nilai_rasio) / nilai_rasio <= tolerance else 'lainnya'

image_df['aspect_ratio_group'] = image_df['aspect_ratio'].map(kelompokkan_rasio)
aspect_ratio_counts = image_df['aspect_ratio_group'].value_counts()
aspect_ratio_summary = pd.DataFrame({'jumlah': aspect_ratio_counts, 'persentase': (aspect_ratio_counts / len(image_df) * 100).round(2)})
display(aspect_ratio_summary)

aspect_ratio_counts.plot.bar(title='Frekuensi Aspect Ratio', xlabel='Rasio', ylabel='Jumlah', figsize=(8, 4))
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

image_df['aspect_ratio'].plot.hist(bins=30, title='Distribusi nilai aspect ratio', xlabel='Width / Height', ylabel='Jumlah', figsize=(7, 4))
plt.tight_layout(); plt.show()

banyak file yang memiliki ukuran standar seperti 9:16 dan 4:3, tapi banyak juga yang memiliki rasio lebih unik lagi.

### e. Ukuran File

In [ ]:
size_lookup = {path.name: path.stat().st_size / 1024 for path in image_files}
image_df['size_kb'] = image_df['filename'].map(size_lookup)
display(image_df['size_kb'].describe().to_frame().T)
image_df['size_kb'].plot.hist(bins=30, title='Distribusi ukuran file', xlabel='Ukuran (KB)', figsize=(7, 4)); plt.tight_layout(); plt.show()

### f. Metadata / EXIF

In [ ]:
exif_records = []
for path in image_files:
    with Image.open(path) as image:
        metadata = {TAGS.get(tag_id, str(tag_id)): value for tag_id, value in image.getexif().items()}
        exif_records.append({'filename': path.name, 'has_exif': bool(metadata), 'camera_make': metadata.get('Make'), 'camera_model': metadata.get('Model'), 'orientation': metadata.get('Orientation'), 'software': metadata.get('Software'), 'timestamp': metadata.get('DateTimeOriginal', metadata.get('DateTime')), 'exif_tags': sorted(metadata.keys())})
exif_df = pd.DataFrame(exif_records)
print(f"Gambar dengan EXIF: {exif_df['has_exif'].sum()} ({exif_df['has_exif'].mean() * 100:.2f}%)")
print(f"Gambar tanpa EXIF: {(~exif_df['has_exif']).sum()} ({(~exif_df['has_exif']).mean() * 100:.2f}%)")
display(exif_df[exif_df['has_exif']])
print('Pola metadata yang tersedia:')
display(exif_df['exif_tags'].value_counts().rename('jumlah gambar').to_frame().head(20))

tidak ada gambar yang memiliki exif(Exchangeable image file format), some kind of metadata gitu.

## 5. Deteksi Batch / Sumber Berbeda

### a. Cross-tab Resolusi × Format

In [ ]:
image_df['resolution_group'] = image_df['width'].astype(str) + ' × ' + image_df['height'].astype(str)
resolution_format = pd.crosstab(image_df['resolution_group'], image_df['actual_format'])
display(resolution_format.sort_values(resolution_format.columns.tolist(), ascending=False).head(20))

print('Resolusi yang paling sering muncul:')
display(image_df['resolution_group'].value_counts().head(15).rename('jumlah').to_frame())

plt.figure(figsize=(10, 5))
image_df.groupby(['resolution_group', 'actual_format']).size().unstack(fill_value=0).head(15).plot.bar(stacked=True, figsize=(10, 5), title='Resolusi × format')
plt.xlabel('Resolusi'); plt.ylabel('Jumlah'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

kebanyakan gambar punya resolusi 4K(2160x3840)

### b. Cross-tab Resolusi × Ukuran File

In [ ]:
image_df['size_bin'] = pd.qcut(image_df['size_kb'], q=4, duplicates='drop')
resolution_size = pd.crosstab(image_df['resolution_group'], image_df['size_bin'])
display(resolution_size.sort_values(resolution_size.columns.tolist(), ascending=False).head(20))

plt.figure(figsize=(10, 5))
resolution_size.head(15).plot.bar(stacked=True, figsize=(10, 5), title='Resolusi × ukuran file')
plt.xlabel('Resolusi'); plt.ylabel('Jumlah'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

plt.figure(figsize=(9, 5))
for image_format, group in image_df.groupby('actual_format'):
    plt.scatter(group['pixels'], group['size_kb'], alpha=0.6, label=image_format)
plt.xscale('log'); plt.yscale('log'); plt.xlabel('Jumlah pixel'); plt.ylabel('Ukuran file (KB)'); plt.title('Resolusi × ukuran file'); plt.legend(); plt.tight_layout(); plt.show()

makes sense, semakin tinggi pixel, semakin tinggi ukuran filenya

### c. Sample Visual per Cluster

In [ ]:
image_df['cluster_key'] = image_df['resolution_group'] + ' | ' + image_df['actual_format'].astype(str) + ' | ' + image_df['size_bin'].astype(str)
cluster_counts = image_df['cluster_key'].value_counts()
print('Cluster terbesar:')
display(cluster_counts.head(10).rename('jumlah').to_frame())

for cluster_name in cluster_counts.head(4).index:
    samples = image_df[image_df['cluster_key'] == cluster_name].sample(min(4, cluster_counts[cluster_name]), random_state=42)
    fig, axes = plt.subplots(1, len(samples), figsize=(14, 3))
    axes = np.atleast_1d(axes)
    fig.suptitle(f'Cluster: {cluster_name}', y=1.05)
    for ax, (_, row) in zip(axes, samples.iterrows()):
        with Image.open(images_dir / row['filename']) as image:
            ax.imshow(image)
        ax.set_title(row['filename'], fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 6. Brightness & Contrast

### a. Hitung Brightness dan Contrast

In [ ]:
brightness_contrast = []
for path in image_files:
    with Image.open(path) as image:
        grayscale = np.asarray(image.convert('L'), dtype=np.float32)
    brightness_contrast.append({
        'filename': path.name,
        'brightness_mean': grayscale.mean(),
        'contrast_std': grayscale.std(),
    })

quality_df = pd.DataFrame(brightness_contrast)
display(quality_df[['brightness_mean', 'contrast_std']].describe())

for context, brightness punya range antara 0(sangat gelap)-255(sangat terang), sedangkan contrast setengahnya, 0(sangat tidak kontras)-127.5(sangat kontras)

### b. Distribusi Brightness dan Contrast

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
quality_df['brightness_mean'].plot.hist(bins=30, ax=axes[0], title='Distribusi brightness', xlabel='Mean grayscale', ylabel='Jumlah')
quality_df['contrast_std'].plot.hist(bins=30, ax=axes[1], title='Distribusi contrast', xlabel='Std grayscale', ylabel='Jumlah')
plt.tight_layout(); plt.show()

### c. Threshold Kasar untuk Kualitas Gambar

In [ ]:
dark_threshold = quality_df['brightness_mean'].quantile(0.10)
bright_threshold = quality_df['brightness_mean'].quantile(0.90)
low_contrast_threshold = quality_df['contrast_std'].quantile(0.10)

quality_df['quality_flags'] = quality_df.apply(
    lambda row: ', '.join([
        label for label, condition in [
            ('terlalu gelap', row['brightness_mean'] < dark_threshold),
            ('terlalu terang', row['brightness_mean'] > bright_threshold),
            ('kontras rendah', row['contrast_std'] < low_contrast_threshold),
        ] if condition
    ]) or 'normal',
    axis=1,
)

print(f'Treshold terlalu gelap  : brightness < {dark_threshold:.2f}')
print(f'Treshold terlalu terang : brightness > {bright_threshold:.2f}')
print(f'Treshold kontras rendah: contrast < {low_contrast_threshold:.2f}')
display(quality_df['quality_flags'].value_counts().rename('jumlah').to_frame())
display(quality_df[quality_df['quality_flags'] != 'normal'].sort_values(['quality_flags', 'brightness_mean']))

cukup banyak file yang secara kecerahan dan kontras normal(tidak 10% terendah maupun 10% tertinggi)

### d. Visual Sample dari Ekor Distribusi

In [ ]:
sample_groups = {
    'paling gelap': quality_df.nsmallest(4, 'brightness_mean'),
    'paling terang': quality_df.nlargest(4, 'brightness_mean'),
    'kontras paling rendah': quality_df.nsmallest(4, 'contrast_std'),
}

for group_name, samples in sample_groups.items():
    fig, axes = plt.subplots(1, len(samples), figsize=(14, 3))
    axes = np.atleast_1d(axes)
    fig.suptitle(group_name, y=1.05)
    for ax, (_, row) in zip(axes, samples.iterrows()):
        with Image.open(images_dir / row['filename']) as image:
            ax.imshow(image)
        ax.set_title(f"{row['filename']}\nB={row['brightness_mean']:.1f}, C={row['contrast_std']:.1f}", fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 7. Blur

### a. Variance of Laplacian

In [ ]:
blur_records = []
for path in image_files:
    with Image.open(path) as image:
        grayscale = np.asarray(image.convert('L'), dtype=np.float32)
    laplacian = laplace(grayscale)
    blur_records.append({'filename': path.name, 'laplacian_variance': laplacian.var()})

blur_df = pd.DataFrame(blur_records)
display(blur_df['laplacian_variance'].describe().to_frame().T)

### b. Distribusi Nilai Blur

In [ ]:
plt.figure(figsize=(8, 4))
blur_df['laplacian_variance'].plot.hist(bins=40, title='Distribusi variance of Laplacian', xlabel='Variance of Laplacian', ylabel='Jumlah')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
blur_df['laplacian_variance'].plot.hist(bins=40, logx=True, title='Distribusi blur dalam skala log', xlabel='Variance of Laplacian (log)', ylabel='Jumlah')
plt.tight_layout(); plt.show()

### c. Threshold Kasar Blur Berat

In [ ]:
blur_threshold = blur_df['laplacian_variance'].quantile(0.10)
blur_df['blur_flag'] = np.where(blur_df['laplacian_variance'] < blur_threshold, 'blur berat', 'tidak terindikasi blur berat')
print(f'Threshold kasar blur berat: variance of Laplacian < {blur_threshold:.2f}')
display(blur_df['blur_flag'].value_counts().rename('jumlah').to_frame())
display(blur_df[blur_df['blur_flag'] == 'blur berat'].sort_values('laplacian_variance'))

### d. Visual Sample Gambar Paling Blur

In [ ]:
blur_samples = blur_df.nsmallest(8, 'laplacian_variance')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, blur_samples.iterrows()):
    with Image.open(images_dir / row['filename']) as image:
        ax.imshow(image)
    ax.set_title(f"{row['filename']}\nVariance={row['laplacian_variance']:.2f}", fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(blur_samples):]:
    ax.axis('off')
plt.suptitle('Sample gambar dengan indikasi blur paling berat')
plt.tight_layout(); plt.show()

sharp_samples = blur_df.nlargest(8, 'laplacian_variance')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, sharp_samples.iterrows()):
    with Image.open(images_dir / row['filename']) as image:
        ax.imshow(image)
    ax.set_title(f"{row['filename']}\nVariance={row['laplacian_variance']:.2f}", fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(sharp_samples):]:
    ax.axis('off')
plt.suptitle('Sample gambar paling tajam')
plt.tight_layout(); plt.show()

## 8. Noise

### a. Residual Std dari Median Filter

In [ ]:
noise_records = []
for path in image_files:
    with Image.open(path) as image:
        grayscale = np.asarray(image.convert('L'), dtype=np.float32)
    smoothed = median_filter(grayscale, size=3)
    residual = grayscale - smoothed
    noise_records.append({'filename': path.name, 'noise_residual_std': residual.std()})

noise_df = pd.DataFrame(noise_records)
display(noise_df['noise_residual_std'].describe().to_frame().T)

### b. Distribusi Noise

In [ ]:
noise_threshold = noise_df['noise_residual_std'].quantile(0.90)
noise_df['noise_flag'] = np.where(noise_df['noise_residual_std'] > noise_threshold, 'noise tinggi', 'normal')
print(f'Threshold kasar noise tinggi: residual std > {noise_threshold:.2f}')
display(noise_df['noise_flag'].value_counts().rename('jumlah').to_frame())

noise_df['noise_residual_std'].plot.hist(bins=30, title='Distribusi residual std noise', xlabel='Residual std', ylabel='Jumlah', figsize=(8, 4))
plt.axvline(noise_threshold, color='red', linestyle='--', label=f'Threshold = {noise_threshold:.2f}')
plt.legend(); plt.tight_layout(); plt.show()

### c. Visual Sample Noise Tertinggi

In [ ]:
noise_samples = noise_df.nlargest(6, 'noise_residual_std')
fig, axes = plt.subplots(len(noise_samples), 2, figsize=(10, 3 * len(noise_samples)))
axes = np.atleast_2d(axes)
for row_index, (_, row) in enumerate(noise_samples.iterrows()):
    with Image.open(images_dir / row['filename']) as image:
        grayscale = np.asarray(image.convert('L'), dtype=np.float32)
    smoothed = median_filter(grayscale, size=3)
    residual = np.abs(grayscale - smoothed)
    axes[row_index, 0].imshow(grayscale, cmap='gray')
    axes[row_index, 0].set_title(f"Asli: {row['filename']}\nNoise std={row['noise_residual_std']:.2f}", fontsize=9)
    axes[row_index, 1].imshow(residual, cmap='inferno', vmin=0, vmax=max(1, np.percentile(residual, 99)))
    axes[row_index, 1].set_title('Residual |original - median|', fontsize=9)
    axes[row_index, 0].axis('off'); axes[row_index, 1].axis('off')
plt.suptitle('Sample noise tertinggi: cek apakah noise atau tekstur/background', y=1.01)
plt.tight_layout(); plt.show()

## 9. Background

Catatan: **EDA bagian ini bersifat eksperimental.** Analisis background_corner_mean dan background_corner_std bergantung pada framing gambar. Karena beberapa gambar memiliki ID card yang memenuhi sebagian besar frame, area sudut tidak selalu merepresentasikan background. Oleh karena itu, kedua metrik ini digunakan sebagai indikator tambahan dan perlu diinterpretasikan sesuai framing masing-masing gambar.

### a. Proxy Background dari Area Corner

In [ ]:
background_records = []
for path in image_files:
    with Image.open(path) as image:
        grayscale = np.asarray(image.convert('L'), dtype=np.float32)
    height, width = grayscale.shape
    patch_height = max(1, int(height * 0.10))
    patch_width = max(1, int(width * 0.10))
    corner_pixels = np.concatenate([
        grayscale[:patch_height, :patch_width].ravel(),
        grayscale[:patch_height, -patch_width:].ravel(),
        grayscale[-patch_height:, :patch_width].ravel(),
        grayscale[-patch_height:, -patch_width:].ravel(),
    ])
    background_records.append({
        'filename': path.name,
        'background_corner_std': corner_pixels.std(),
        'background_corner_mean': corner_pixels.mean(),
    })

background_df = pd.DataFrame(background_records)
display(background_df[['background_corner_std', 'background_corner_mean']].describe())

### b. Distribusi Background

In [ ]:
background_threshold = background_df['background_corner_std'].quantile(0.90)
background_df['background_flag'] = np.where(background_df['background_corner_std'] > background_threshold, 'background ramai', 'normal')
print(f'Threshold kasar background ramai: corner std > {background_threshold:.2f}')
display(background_df['background_flag'].value_counts().rename('jumlah').to_frame())

background_df['background_corner_std'].plot.hist(bins=30, title='Distribusi std area corner', xlabel='Std pixel corner', ylabel='Jumlah', figsize=(8, 4))
plt.axvline(background_threshold, color='red', linestyle='--', label=f'Threshold = {background_threshold:.2f}')
plt.legend(); plt.tight_layout(); plt.show()

### c. Visual Sample Background Paling Ramai

In [ ]:
background_samples = background_df.nlargest(8, 'background_corner_std')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, background_samples.iterrows()):
    with Image.open(images_dir / row['filename']) as image:
        ax.imshow(image)
    ax.set_title(f"{row['filename']}\nCorner std={row['background_corner_std']:.2f}", fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(background_samples):]:
    ax.axis('off')
plt.suptitle('Sample background dengan variasi pixel paling tinggi', y=1.02)
plt.tight_layout(); plt.show()

## 10. Orientation & Rotation (Manual/Visual)

### a. Orientasi dari Aspect Ratio

In [ ]:
def kategori_orientasi(value, tolerance=0.03):
    if abs(value - 1) / 1 <= tolerance:
        return 'square'
    return 'landscape' if value > 1 else 'portrait'

image_df['orientation_group'] = image_df['aspect_ratio'].map(kategori_orientasi)
orientation_counts = image_df['orientation_group'].value_counts()
orientation_summary = pd.DataFrame({'jumlah': orientation_counts, 'persentase': (orientation_counts / len(image_df) * 100).round(2)})
display(orientation_summary)

orientation_counts.plot.bar(title='Orientasi kasar berdasarkan aspect ratio', xlabel='Orientasi', ylabel='Jumlah', figsize=(7, 4))
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### b. Cek Minoritas Orientasi

In [ ]:
minority_orientation = orientation_counts.idxmin()
minority_images = image_df[image_df['orientation_group'] == minority_orientation]
print(f"Orientasi minoritas: {minority_orientation} ({len(minority_images)} gambar)")
display(minority_images[['filename', 'width', 'height', 'aspect_ratio', 'orientation_group']].head(20))

### c. Random Sampling untuk Cek Miring / Rotated

In [ ]:
sample_size = min(40, len(image_df))
orientation_sample = image_df.sample(sample_size, random_state=42)
print(f"Menampilkan {sample_size} gambar untuk inspeksi visual.")

fig, axes = plt.subplots(5, 8, figsize=(16, 12))
for ax, (_, row) in zip(axes.flat, orientation_sample.iterrows()):
    with Image.open(images_dir / row['filename']) as image:
        ax.imshow(image)
    ax.set_title(row['filename'], fontsize=7)
    ax.axis('off')
for ax in axes.flat[sample_size:]:
    ax.axis('off')
plt.suptitle('Random sample untuk cek visual orientasi dan kemiringan', y=0.995)
plt.tight_layout(); plt.show()

## 11. Korelasi Kualitas Gambar vs Ground Truth

### a. Gabungkan Quality Metrics dan Ground Truth

In [ ]:
quality_metrics = image_df.merge(quality_df, on='filename', how='left')
quality_metrics = quality_metrics.merge(blur_df, on='filename', how='left')
quality_metrics = quality_metrics.merge(noise_df, on='filename', how='left')
quality_metrics = quality_metrics.merge(background_df, on='filename', how='left')

ground_truth_join = quality_metrics.merge(df, on='filename', how='left', indicator='ground_truth_status')
ground_truth_join['has_ground_truth'] = ground_truth_join['ground_truth_status'].eq('both')
ground_truth_join['address_empty'] = ground_truth_join['address'].fillna('').astype(str).str.strip().eq('')

print(f"Total gambar: {len(ground_truth_join)}")
print(f"Dengan ground truth: {ground_truth_join['has_ground_truth'].sum()}")
print(f"Tanpa ground truth: {(~ground_truth_join['has_ground_truth']).sum()}")
display(ground_truth_join.head())

### b. Address Kosong vs Kualitas Gambar

In [ ]:
quality_columns = ['brightness_mean', 'contrast_std', 'laplacian_variance', 'noise_residual_std', 'background_corner_std', 'size_kb']
address_quality_summary = ground_truth_join[ground_truth_join['has_ground_truth']].groupby('address_empty')[quality_columns].agg(['count', 'mean', 'median'])
display(address_quality_summary)

for metric in ['brightness_mean', 'contrast_std', 'noise_residual_std', 'background_corner_std']:
    ground_truth_join.boxplot(column=metric, by='address_empty', figsize=(6, 4))
    plt.title(f'{metric} berdasarkan address kosong/tidak'); plt.suptitle(''); plt.xlabel('Address kosong'); plt.tight_layout(); plt.show()

### c. Konsistensi Kualitas pada Identitas Duplikat

In [ ]:
ground_truth_join['identity_key'] = ground_truth_join[['name', 'birth_date', 'address']].fillna('').astype(str).agg(' | '.join, axis=1)
duplicate_identity_keys = ground_truth_join['identity_key'].value_counts()
duplicate_identity_keys = duplicate_identity_keys[duplicate_identity_keys > 1].index
duplicate_quality = ground_truth_join[ground_truth_join['identity_key'].isin(duplicate_identity_keys)].copy()

duplicate_spread = duplicate_quality.groupby('identity_key')[quality_columns].agg(lambda values: values.max() - values.min()).sort_values('laplacian_variance', ascending=False)
print(f"Identitas dengan lebih dari satu gambar: {len(duplicate_identity_keys)}")
print('Rentang kualitas antar gambar dalam identitas yang sama:')
display(duplicate_spread.head(20))

if len(duplicate_identity_keys):
    example_key = duplicate_spread.index[0]
    display(duplicate_quality[duplicate_quality['identity_key'] == example_key][['filename'] + quality_columns])

### d. Filename Tanpa Ground Truth

In [ ]:
missing_gt_summary = ground_truth_join.groupby('has_ground_truth')[quality_columns].agg(['count', 'mean', 'median'])
display(missing_gt_summary)

for metric in ['brightness_mean', 'contrast_std', 'laplacian_variance', 'noise_residual_std', 'background_corner_std']:
    ground_truth_join.boxplot(column=metric, by='has_ground_truth', figsize=(6, 4))
    plt.title(f'{metric}: ada ground truth vs tidak'); plt.suptitle(''); plt.xlabel('Ada ground truth'); plt.tight_layout(); plt.show()

### e. Missing Value vs Batch / Source

In [ ]:
missing_flags = ground_truth_join[['filename', 'cluster_key', 'name', 'birth_date', 'address']].copy()
for field in ['name', 'birth_date', 'address']:
    missing_flags[f'{field}_missing'] = missing_flags[field].isna() | missing_flags[field].astype(str).str.strip().eq('')

missing_by_cluster = missing_flags.groupby('cluster_key')[[f'{field}_missing' for field in ['name', 'birth_date', 'address']]].mean().mul(100).round(2)
display(missing_by_cluster.sort_values(missing_by_cluster.columns.tolist(), ascending=False).head(20))
print('Catatan: cluster hanya dipakai untuk audit pola batch/source, bukan sebagai fitur OCR.')

## 12. Ringkasan Kuantitatif

### a. Persentase Gambar per Kategori Masalah

In [ ]:
issue_summary = pd.DataFrame({
    'format_mismatch': image_df['format_mismatch'].astype(bool),
    'blur_berat': blur_df['blur_flag'].eq('blur berat'),
    'noise_tinggi': noise_df['noise_flag'].eq('noise tinggi'),
    'background_ramai': background_df['background_flag'].eq('background ramai'),
    'terlalu_gelap': quality_df['quality_flags'].str.contains('terlalu gelap'),
    'terlalu_terang': quality_df['quality_flags'].str.contains('terlalu terang'),
    'kontras_rendah': quality_df['quality_flags'].str.contains('kontras rendah'),
}).agg(['sum', 'mean']).T
issue_summary.columns = ['jumlah', 'proporsi']
issue_summary['persentase'] = (issue_summary['proporsi'] * 100).round(2)
display(issue_summary.drop(columns='proporsi').sort_values('persentase', ascending=False))

issue_counts_per_image = issue_summary[['jumlah']].copy()
all_issue_flags = pd.DataFrame({
    'format_mismatch': image_df['format_mismatch'].astype(bool),
    'blur_berat': blur_df['blur_flag'].eq('blur berat'),
    'noise_tinggi': noise_df['noise_flag'].eq('noise tinggi'),
    'background_ramai': background_df['background_flag'].eq('background ramai'),
    'terlalu_gelap': quality_df['quality_flags'].str.contains('terlalu gelap'),
    'terlalu_terang': quality_df['quality_flags'].str.contains('terlalu terang'),
    'kontras_rendah': quality_df['quality_flags'].str.contains('kontras rendah'),
})
print(f"Gambar dengan minimal satu flag kualitas: {(all_issue_flags.sum(axis=1) > 0).sum()} dari {len(all_issue_flags)} ({(all_issue_flags.sum(axis=1) > 0).mean() * 100:.2f}%)")

### b. Ringkasan Semua Metrik

In [ ]:
summary_metric_columns = [
    'width', 'height', 'pixels', 'aspect_ratio', 'size_kb',
    'brightness_mean', 'contrast_std', 'laplacian_variance',
    'noise_residual_std', 'background_corner_std', 'background_corner_mean',
]
all_metrics_summary = ground_truth_join[summary_metric_columns].describe().T
display(all_metrics_summary)

### c. Temuan yang Saling Terhubung

In [ ]:
print('Temuan lintas tahap:')
print(f"- Total file yang dianalisis: {len(image_df)}; file corrupt: {len(image_files) - len(image_df)}.")
print(f"- Format mismatch: {int(issue_summary.loc['format_mismatch', 'jumlah'])} file.")
print(f"- Identitas duplikat: {len(identity_duplicate_keys)} grup; grup byte-identik: {len(duplicate_hash_groups)} grup.")
print(f"- Gambar blur berat: {int(issue_summary.loc['blur_berat', 'jumlah'])}; noise tinggi: {int(issue_summary.loc['noise_tinggi', 'jumlah'])}.")
print(f"- Background ramai: {int(issue_summary.loc['background_ramai', 'jumlah'])}; kontras rendah: {int(issue_summary.loc['kontras_rendah', 'jumlah'])}.")
print(f"- Ground truth tersedia untuk {int(ground_truth_join['has_ground_truth'].sum())} dari {len(ground_truth_join)} gambar.")
print('Gunakan sample visual dari section 5–11 untuk memastikan flag statistik tidak keliru membaca tekstur, crop, atau karakter dokumen sebagai masalah kualitas.')